# 05 · Unity Catalog — gobierno de datos

Unity Catalog centraliza **permisos, linaje, descubrimiento y clasificación** de todos los activos.
En CFE permite exponer la capa **oro** a otras herramientas de forma gobernada.

En este lab: jerarquía de objetos, permisos (RBAC), `information_schema`, tags de clasificación,
linaje, y máscaras de columna para datos sensibles (montos de contrato).

> Requiere haber corrido el lab **01** (usa tu tabla `delta_obras`).

In [0]:
%run ./_setup

## 1. Jerarquía: catálogo › esquema › tabla
Todo objeto se referencia por su nombre de 3 niveles `catalog.schema.table`.

In [0]:
display(spark.sql(f"SHOW TABLES IN `{CATALOG}`.`{WORK_SCHEMA}`"))

## 2. Permisos (RBAC) — GRANT / REVOKE
```sql
GRANT USE CATALOG ON CATALOG dipidev TO `analistas_construccion`;
GRANT USE SCHEMA, SELECT ON SCHEMA <tu_esquema> TO `analistas_construccion`;
GRANT SELECT ON TABLE <tu_esquema>.gold_obras_por_estado TO `analistas_construccion`;
REVOKE SELECT ON TABLE <tu_esquema>.silver_obras_cfe FROM `analistas_construccion`;
```

In [0]:
display(spark.sql(f"SHOW GRANTS ON TABLE {tbl('delta_obras')}"))

## 3. `information_schema` — metadatos consultables con SQL

In [0]:
display(spark.sql(f"""
  SELECT table_name, table_type, created, last_altered
  FROM `{CATALOG}`.information_schema.tables
  WHERE table_schema = '{WORK_SCHEMA}'
  ORDER BY created DESC
"""))

## 4. Tags de clasificación
Etiqueta objetos/columnas para gobierno (capa medallion, sensibilidad, dominio).

In [0]:
spark.sql(f"ALTER TABLE {tbl('delta_obras')} SET TAGS ('capa' = 'bronze', 'dominio' = 'construccion')")
spark.sql(f"ALTER TABLE {tbl('delta_obras')} ALTER COLUMN ImporteContratoMxn SET TAGS ('sensibilidad' = 'alta', 'tipo_dato' = 'financiero')")
spark.sql(f"ALTER TABLE {tbl('delta_obras')} ALTER COLUMN NombreObra SET TAGS ('tipo_dato' = 'descriptivo')")

display(spark.sql(f"""
  SELECT tag_name, tag_value FROM `{CATALOG}`.information_schema.table_tags
  WHERE schema_name = '{WORK_SCHEMA}' AND table_name = 'delta_obras'
"""))

## 4b. Asignar tags desde Catalog Explorer (interfaz gráfica)

Además de SQL, puedes etiquetar tablas y columnas directamente desde la UI.

### Etiquetar una tabla
1. En el menú lateral izquierdo, haz clic en **Catalog** (icono de base de datos)
2. Navega: `dipidev` → `default` → `delta_obras`
3. En la pestaña **Overview**, busca la sección **Tags**
4. Haz clic en **✚ Add tag**
5. Escribe la **clave** (ej: `capa`) y el **valor** (ej: `bronze`)
6. Haz clic en **Add** para confirmar

### Etiquetar una columna
1. En la misma tabla, ve a la pestaña **Columns** (o **Schema**)
2. Haz clic en la columna que quieras etiquetar (ej: `ImporteContratoMxn`)
3. En el panel de detalle de la columna, busca **Tags**
4. Haz clic en **✚ Add tag**
5. Escribe clave: `sensibilidad`, valor: `alta`
6. Confirma con **Add**

### ✅ Ejercicio
Desde la UI, agrega estos tags a `delta_obras`:

| Nivel | Clave | Valor |
|---|---|---|
| Tabla | `dominio` | `construccion` |
| Tabla | `responsable` | `tu_nombre` |
| Columna `ImporteContratoMxn` | `sensibilidad` | `alta` |
| Columna `ImporteContratoMxn` | `pii` | `financiero` |
| Columna `NombreObra` | `tipo_dato` | `descriptivo` |

Después, verifica tus tags con la celda de arriba o con esta consulta:
```sql
SELECT * FROM dipidev.information_schema.column_tags
WHERE table_name = 'delta_obras'
```

## 5b. Aplicar política de enmascaramiento desde Catalog Explorer

Las **column masks** ocultan datos sensibles (montos, nombres, RFC) según el grupo del usuario.
Se necesitan dos pasos: crear la función de máscara y aplicarla a la columna.

### Paso 1 — Crear la función de máscara (SQL, una sola vez)
Ejecuta la siguiente celda para crear la función. Solo los miembros del grupo `finanzas_cfe`
verán el valor real; el resto verá `NULL`.

### Paso 2 — Aplicar la máscara desde la UI
1. En **Catalog Explorer**, navega a `dipidev` → `default` → `delta_obras`
2. Ve a la pestaña **Columns** (o **Schema**)
3. Haz clic en la columna `ImporteContratoMxn`
4. En el panel de detalle, busca la sección **Column mask**
5. Haz clic en **Add column mask**
6. En el diálogo:
   * **Function catalog**: `dipidev`
   * **Function schema**: `default`
   * **Function name**: selecciona `mask_importe`
7. Haz clic en **Apply** para activar la máscara

### ¿Cómo verificar?
* Los usuarios **fuera** del grupo `finanzas_cfe` verán `NULL` en `ImporteContratoMxn`
* Los usuarios **dentro** del grupo verán el valor real
* En la UI, la columna mostrará un icono de 🔒 candado indicando que tiene máscara

### Para remover la máscara (desde la UI)
1. Haz clic en la columna `ImporteContratoMxn`
2. En **Column mask**, haz clic en **Remove**

### Para remover la máscara (SQL)
```sql
ALTER TABLE dipidev.default.delta_obras
  ALTER COLUMN ImporteContratoMxn DROP MASK;
```

In [0]:
%sql
-- Paso 1: Crear la función de máscara
-- Solo los miembros de 'finanzas_cfe' ven el importe real
CREATE OR REPLACE FUNCTION dipidev.default.mask_importe(valor DOUBLE)
  RETURN CASE
    WHEN is_account_group_member('finanzas_cfe') THEN valor
    ELSE NULL
  END

In [0]:
%sql
-- Verifica: ¿qué ve un usuario sin el grupo finanzas_cfe?
-- (Si tú estás en el grupo, verás el valor real)
SELECT IdObra, NombreObra, ImporteContratoMxn
FROM dipidev.default.delta_obras
LIMIT 5

## 5. Linaje
UC captura linaje automáticamente. Abre tu tabla `delta_obras` en **Catalog Explorer → Lineage**.

## 6. (Avanzado) Máscara de columna para datos financieros
```sql
CREATE OR REPLACE FUNCTION <tu_esquema>.mask_importe(v DOUBLE)
  RETURN CASE WHEN is_account_group_member('finanzas_cfe') THEN v ELSE -1.0 END;

ALTER TABLE <tu_esquema>.delta_obras
  ALTER COLUMN ImporteContratoMxn SET MASK <tu_esquema>.mask_importe;
```